In [1]:
import plotly.graph_objects as go
import numpy as np

# --- データの準備 (G7 + China, 2023-2024 GDP) ---
# 色設定:
# CHN(中国) = 赤 (Red)
# JPN(日本) = 白/銀 (Silver) - 中国の赤と区別し、国旗の地色かつ「極東の技術ハブ」を表現
# DEU(ドイツ) = 緑 (Green)
# ...他は前回踏襲
geo_data = {
    "USA": {"coords": (38.90, -77.03), "gdp": 27.36, "rgb": (31, 119, 180)},   # Blue
    "CHN": {"coords": (39.90, 116.40), "gdp": 17.79, "rgb": (214, 39, 40)},    # Red (New!)
    "DEU": {"coords": (52.52, 13.40),  "gdp": 4.46,  "rgb": (44, 160, 44)},    # Green
    "JPN": {"coords": (35.67, 139.65), "gdp": 4.21,  "rgb": (220, 220, 220)},  # Silver (Changed from Red)
    "GBR": {"coords": (51.50, -0.12),  "gdp": 3.34,  "rgb": (148, 103, 189)},  # Purple
    "FRA": {"coords": (48.85, 2.35),   "gdp": 3.03,  "rgb": (0, 0, 128)},      # Navy
    "ITA": {"coords": (41.90, 12.49),  "gdp": 2.25,  "rgb": (188, 189, 34)},   # Olive
    "CAN": {"coords": (45.42, -75.69), "gdp": 2.14,  "rgb": (255, 127, 14)}    # Orange
}

def latlon_to_xyz(lat, lon):
    phi = np.radians(90 - lat)
    theta = np.radians(lon)
    return np.array([np.sin(phi) * np.cos(theta), np.sin(phi) * np.sin(theta), np.cos(phi)])

# --- ジオデシック球（Icosphere）生成ロジック ---
def normalize(v):
    norm = np.linalg.norm(v)
    if norm == 0: return v
    return v / norm

def subdivision(verts, faces):
    new_faces = []
    midpoint_cache = {}

    def get_midpoint(i1, i2):
        key = tuple(sorted((i1, i2)))
        if key in midpoint_cache: return midpoint_cache[key]
        mid = normalize(verts[i1] + verts[i2])
        verts.append(mid)
        idx = len(verts) - 1
        midpoint_cache[key] = idx
        return idx

    for f in faces:
        v1, v2, v3 = f
        a = get_midpoint(v1, v2)
        b = get_midpoint(v2, v3)
        c = get_midpoint(v3, v1)
        new_faces.extend([(v1, a, c), (v2, b, a), (v3, c, b), (a, b, c)])

    return new_faces

# 正二十面体の定義
phi = (1 + np.sqrt(5)) / 2
verts = [
    normalize(np.array([-1, phi, 0])), normalize(np.array([1, phi, 0])),
    normalize(np.array([-1, -phi, 0])), normalize(np.array([1, -phi, 0])),
    normalize(np.array([0, -1, phi])), normalize(np.array([0, 1, phi])),
    normalize(np.array([0, -1, -phi])), normalize(np.array([0, 1, -phi])),
    normalize(np.array([phi, 0, -1])), normalize(np.array([phi, 0, 1])),
    normalize(np.array([-phi, 0, -1])), normalize(np.array([-phi, 0, 1]))
]
faces = [
    (0, 11, 5), (0, 5, 1), (0, 1, 7), (0, 7, 10), (0, 10, 11),
    (1, 5, 9), (5, 11, 4), (11, 10, 2), (10, 7, 6), (7, 1, 8),
    (3, 9, 4), (3, 4, 2), (3, 2, 6), (3, 6, 8), (3, 8, 9),
    (4, 9, 5), (2, 4, 11), (6, 2, 10), (8, 6, 7), (9, 8, 1)
]

# 再帰的細分化 (解像度)
subdivisions = 5  # 少し細かくして境界を綺麗に
for _ in range(subdivisions):
    faces = subdivision(verts, faces)

verts = np.array(verts)
faces = np.array(faces)

# --- ポリゴンの所有権判定 ---
face_colors = []
country_names = list(geo_data.keys())

for face in faces:
    v_coords = verts[face]
    centroid = normalize(np.mean(v_coords, axis=0))

    min_weighted_dist = np.inf
    dominant_country_idx = 0

    for i, name in enumerate(country_names):
        d = geo_data[name]
        c_xyz = latlon_to_xyz(*d["coords"])

        # 測地線距離
        dist = np.arccos(np.clip(np.dot(centroid, c_xyz), -1, 1))

        # 【統治工学式】 自律重み付け
        # 中国の巨大質量が入っても、日本の領域が消滅しないかを確認する重要な式
        weighted_dist = dist / (d["gdp"] ** 0.12)

        if weighted_dist < min_weighted_dist:
            min_weighted_dist = weighted_dist
            dominant_country_idx = i

    c_rgb = geo_data[country_names[dominant_country_idx]]["rgb"]
    face_colors.append(f'rgb({c_rgb[0]}, {c_rgb[1]}, {c_rgb[2]})')

# --- 描画 ---
fig = go.Figure()

fig.add_trace(go.Mesh3d(
    x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
    i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
    facecolor=face_colors,
    flatshading=True,
    name="Geodesic World",
    lighting=dict(ambient=0.6, diffuse=0.8, roughness=0.1),
    lightposition=dict(x=100, y=100, z=1000)
))

# 母点プロット
for name, d in geo_data.items():
    c_xyz = latlon_to_xyz(*d["coords"])
    fig.add_trace(go.Scatter3d(
        x=[c_xyz[0]*1.05], y=[c_xyz[1]*1.05], z=[c_xyz[2]*1.05],
        mode='markers+text',
        marker=dict(size=5, color='white', line=dict(width=2, color='black')),
        text=name, textposition="top center",
        textfont=dict(color='white', size=10),
        name=f"{name} ({d['gdp']}T)"
    ))

fig.update_layout(
    title=dict(
        text="<b>Geodesic Polyhedron Model (G7 + China)</b><br>2024 GDP Estimates: The Three-Body Problem in Pacific",
        x=0.5, font=dict(size=16, color="white")
    ),
    scene=dict(
        xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False),
        aspectmode='data', bgcolor='rgb(10,10,15)'
    ),
    width=900, height=800, margin=dict(r=0, l=0, b=0, t=60),
    paper_bgcolor='rgb(10,10,15)',
    showlegend=True,
    legend=dict(font=dict(color="white"), x=0.8, y=0.5)
)

fig.show()